# Ensemble Learning

## What is Ensemble Learning?
Ensemble learning is an advanced machine learning technique where multiple individual models (often called weak learners) are combined together to create a single, highly powerful and accurate prediction model. 

The core idea is inspired by the **"Wisdom of the Crowd"**—relying on the collective decision of a group of experts rather than trusting a single individual.

---

## Why Do We Need Ensemble Methods?
* **Higher Accuracy:** Combining models usually yields significantly better predictive performance than any single model alone.
* **Error Reduction:** Individual models make different types of errors; combining them helps cancel out these individual mistakes.
* **Stability and Robustness:** Ensemble models are less sensitive to noise and outliers in the dataset.

---

## Main Types of Ensemble Techniques

### 1. Bagging (Bootstrap Aggregating)
* **Objective:** Designed to reduce variance and prevent overfitting.
* **How it works:** Creates multiple subsets of the original dataset with replacement. A separate base model (usually a Decision Tree) is trained independently and in parallel on each subset.
* **Final Output:** Averages the predictions (for regression) or takes a majority vote (for classification).
* **Famous Example:** Random Forest.

### 2. Boosting
* **Objective:** Designed to reduce bias and convert weak models into strong ones.
* **How it works:** Trains models sequentially (one after another). Each new model explicitly focuses on fixing the errors and misclassifications made by the previous model.
* **Famous Examples:** Gradient Boosting (GBM), XGBoost, LightGBM, and AdaBoost.

### 3. Stacking (Stacked Generalization)
* **How it works:** Combines entirely different types of algorithms (e.g., KNN, SVM, Decision Tree) as base models. A secondary model, known as a **Meta-Model** or blender, takes the predictions of the base models as its input to make the final decision.

### 4. Voting Classifiers
* **Hard Voting:** All models vote for a class, and the class with the most votes wins (majority rules).
* **Soft Voting:** Models output probabilities, and the final decision is based on the average probability score across all models.

---

## Advantages of Ensemble Methods
* Superior prediction accuracy compared to single models.
* Exceptional handling of complex, non-linear relationships.
* High resistance to overfitting (especially with Bagging/Random Forests).

## Disadvantages of Ensemble Methods
* **Computationally Expensive:** Requires more memory and training time because multiple models are trained simultaneously or sequentially.
* **Slower Inference:** Making predictions can take slightly longer due to passing data through multiple underlying models.
* **Lower Interpretability:** Harder to explain the internal decision logic to stakeholders compared to a single Decision Tree.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, recall_score, precision_score, f1_score



## Load data

In [2]:
df = pd.read_csv("C:/Users/hp/Desktop/Machine_Learning/data.csv")
print("Success! Data loaded.")

df = df.drop_duplicates()



C:\Users\hp\AppData\Local\Temp\ipykernel_3504\2133902529.py:1: DtypeWarning: Columns (0: coupon_code, 1: additional) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/hp/Desktop/Machine_Learning/data.csv")


Success! Data loaded.


### Drop columns 

In [3]:
useless_cols = ['cashback', 'tabby_payment_id', 'tabby_order_status', 'paymob_intention_id', 'paymob_cents_amount', 'tamara_order_id', 'tamara_checkout_id', 'cko_hpp_id', 'cko_payment_id', 'cko_session_id', 'id', 'old_order_id', 'increment_id', 'customer_email', 'customer_first_name', 'customer_last_name', 'shipping_title', 'shipping_description', 'coupon_code', 'created_at', 'updated_at']
df_clean = df.drop(columns=[col for col in useless_cols if col in df.columns])

df_clean['target'] = df_clean['status'].apply(lambda x: 0 if str(x).strip().lower() == 'completed' else 1)

df_clean['discount_ratio'] = df_clean['discount_amount'] / (df_clean['sub_total'] + 1e-5)
df_clean['tax_ratio'] = df_clean['tax_amount'] / (df_clean['sub_total'] + 1e-5)
df_clean['shipping_ratio'] = df_clean['shipping_amount'] / (df_clean['grand_total'] + 1e-5)
df_clean['avg_item_price'] = df_clean['sub_total'] / (df_clean['total_qty_ordered'] + 1e-5)



## Feature

In [4]:
core_features = ['sub_total', 'discount_amount', 'tax_amount', 'shipping_amount', 'cod_charges', 'wallet_credit_used', 'total_item_count', 'total_qty_ordered', 'cashback_earned', 'mgmt_fee', 'discount_ratio', 'tax_ratio', 'shipping_ratio', 'avg_item_price', 'channel_name', 'customer_type', 'shipping_method', 'is_guest', 'is_gift', 'cashback_opt_in']
available_features = [col for col in core_features if col in df_clean.columns]

X_raw = df_clean[available_features].copy()
y = df_clean['target']

num_cols = X_raw.select_dtypes(include=['number']).columns
X_raw[num_cols] = X_raw[num_cols].fillna(X_raw[num_cols].median())

cat_cols = X_raw.select_dtypes(include=['object', 'string', 'category']).columns
X_raw[cat_cols] = X_raw[cat_cols].fillna('Missing')

X = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)



### train and Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    min_samples_split=50, 
    class_weight='balanced', 
    n_jobs=-1, 
    random_state=42
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)



### Evaluate 

In [6]:
print("=" * 50)
print(" RANDOM FOREST ENSEMBLE PERFORMANCE ")
print("=" * 50)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("=" * 50)

 RANDOM FOREST ENSEMBLE PERFORMANCE 
Accuracy Score: 0.7551

Classification Report:
               precision    recall  f1-score   support

           0       0.59      0.75      0.66      9247
           1       0.87      0.76      0.81     19994

    accuracy                           0.76     29241
   macro avg       0.73      0.75      0.73     29241
weighted avg       0.78      0.76      0.76     29241

